In [ ]:
%load_ext autoreload
%autoreload 2
%cd /mnt/bd/janne-research-sm/samantha

In [ ]:
# import pandas as pd
# df = pd.read_parquet("hdfs://harunava/home/byte_speech_sv/mulan/long_form_playlist_description_v5_music_id_agg.csv/")

In [ ]:
# df.playlist_names.values[15]

In [ ]:
from recipes.research.dataset.collection import PlaylistV5ParquetDataset
from samantha.data.audio.dataset import AudioFolderDataModule

batch_size = 8
num_workers = 16
sample_rate = 44100
segment_duration = 30

playlist_v5 = PlaylistV5ParquetDataset(
    sample_rate=sample_rate,
    channels=2,
    segment_duration=segment_duration,
    resampled=True,
    shardshuffle=True,
)
datamodule = AudioFolderDataModule([playlist_v5], [], [], weights=None, batch_size=batch_size, shuffle=None, num_workers=num_workers)
train_loader = datamodule.train_dataloader()

In [ ]:
train_loader = iter(train_loader)

In [ ]:
from IPython.display import display, Audio

batch = next(train_loader)
batch_idx = 0
n_frames = batch.segment_info[batch_idx].n_frames
display(Audio(batch.audio[batch_idx, :, :n_frames], rate=sample_rate))

In [ ]:
from samantha.transforms.audio import batch_plot_spectrogram, MelSpectrogram


mel_transform = MelSpectrogram(
    sample_rate=sample_rate,
    n_mels=160,
    n_fft=4096,
).to(batch.audio.device)

mel, _ = mel_transform(batch.audio.mean(dim=1))
batch_plot_spectrogram(mel.cpu(), plot_log=True, figsize=(20, 20))